## Load model and setup

In [10]:
import os
import time
import copy
from pathlib import Path

import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM

if torch.backends.mps.is_available():
    train_device = "mps"
else:
    train_device = "cpu"

quant_device = "cpu"

print("Train device:", train_device)
print("Quantized benchmark device:", quant_device)

finetuned_model_path = "../models/finetuned-gpt2"

tokenizer = AutoTokenizer.from_pretrained(finetuned_model_path)
tokenizer.pad_token = tokenizer.eos_token

fp32_model = AutoModelForCausalLM.from_pretrained(finetuned_model_path)
fp32_model.config.pad_token_id = fp32_model.config.eos_token_id
fp32_model.eval()

print("Loaded fine-tuned model from:", finetuned_model_path)

Train device: mps
Quantized benchmark device: cpu
Loaded fine-tuned model from: ../models/finetuned-gpt2


## Applying dynamic quantization

In [11]:

if "qnnpack" in torch.backends.quantized.supported_engines:
    torch.backends.quantized.engine = "qnnpack"
else:
    raise RuntimeError("qnnpack not available in this environment.")

print("After:", torch.backends.quantized.engine)
quantized_model = torch.quantization.quantize_dynamic(
    fp32_model,            # input model
    {torch.nn.Linear},     # layers to quantize
    dtype=torch.qint8
)
quantized_model.eval()

print("Quantization complete.")

After: qnnpack


/var/folders/dx/6_vmkdws4b12jknf397g1lh80000gn/T/ipykernel_2761/3714101107.py:7: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


Quantization complete.


## Saving quantized model

In [12]:
quant_model_dir = Path("../models/quantized-gpt2")
quant_model_dir.mkdir(parents=True, exist_ok=True)

quant_weights_path = quant_model_dir / "pytorch_model_quantized.bin"
torch.save(quantized_model.state_dict(), quant_weights_path)

tokenizer.save_pretrained(quant_model_dir)

print("Saved quantized weights to:", quant_weights_path)
print("Saved tokenizer to:", quant_model_dir)

Saved quantized weights to: ../models/quantized-gpt2/pytorch_model_quantized.bin
Saved tokenizer to: ../models/quantized-gpt2


## Comparing model size

In [13]:
ft_dir = Path("../models/finetuned-gpt2")
q_dir = Path("../models/quantized-gpt2")

def first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None

# HF model may be safetensors or bin
fp32_file = first_existing([
    ft_dir / "model.safetensors",
    ft_dir / "pytorch_model.bin",
])

quant_file = q_dir / "pytorch_model_quantized.bin"

print("fp32 file:", fp32_file)
print("quant file:", quant_file if quant_file.exists() else None)

def mb(p): return p.stat().st_size / (1024**2)

if fp32_file:
    fp32_size_mb = mb(fp32_file)
    print("FP32 size (MB):", fp32_size_mb)
else:
    fp32_size_mb = None
    print("FP32 file not found")

if quant_file.exists():
    quant_size_mb = mb(quant_file)
    print("Quant size (MB):", quant_size_mb)
else:
    quant_size_mb = None
    print("Quant file not found")

fp32 file: ../models/finetuned-gpt2/model.safetensors
quant file: ../models/quantized-gpt2/pytorch_model_quantized.bin
FP32 size (MB): 474.7144775390625
Quant size (MB): 511.7562093734741


## Exporting to ONNX

In [14]:
from pathlib import Path
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM

ft_model_path = "../models/finetuned-gpt2"
onnx_dir = Path("../models/onnx-gpt2")
onnx_dir.mkdir(parents=True, exist_ok=True)
onnx_fp32_path = onnx_dir / "gpt2_fp32.onnx"

tokenizer = AutoTokenizer.from_pretrained(ft_model_path)
base_model = AutoModelForCausalLM.from_pretrained(ft_model_path)
base_model.config.use_cache = False
base_model.eval()

class GPT2NoCacheWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, input_ids, attention_mask):
        return self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=False
        )[0]  # logits

model = GPT2NoCacheWrapper(base_model).eval()

sample = tokenizer("hello export", return_tensors="pt")
input_ids = sample["input_ids"]
attention_mask = sample["attention_mask"]

with torch.no_grad():
    torch.onnx.export(
        model,
        (input_ids, attention_mask),
        str(onnx_fp32_path),
        input_names=["input_ids", "attention_mask"],
        output_names=["logits"],
        dynamic_axes={
            "input_ids": {0: "batch", 1: "sequence"},
            "attention_mask": {0: "batch", 1: "sequence"},
            "logits": {0: "batch", 1: "sequence"},
        },
        opset_version=17,
        do_constant_folding=False,
        dynamo=False,
        training=torch.onnx.TrainingMode.EVAL,
    )

print("Exported:", onnx_fp32_path)

/var/folders/dx/6_vmkdws4b12jknf397g1lh80000gn/T/ipykernel_2761/137318200.py:35: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/Users/paripurna/Documents/Learning/gen_ai_learning/genai-learning/projects/project-4-fine-tuning-gen-model-quantization/venv311/lib/python3.11/site-packages/transformers/modeling_attn_mask_utils.py:116: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if (input_shape[-1] > 1 or self.sliding_window 

Exported: ../models/onnx-gpt2/gpt2_fp32.onnx


## Quantize to INT8

In [15]:
from onnxruntime.quantization import quantize_dynamic, QuantType

onnx_dir = Path("../models/onnx-gpt2")
onnx_fp32_path = onnx_dir / "gpt2_fp32.onnx"
onnx_int8_path = onnx_dir / "gpt2_int8.onnx"

quantize_dynamic(
    model_input=str(onnx_fp32_path),
    model_output=str(onnx_int8_path),
    weight_type=QuantType.QInt8
)

print("Created INT8 ONNX:", onnx_int8_path)

Created INT8 ONNX: ../models/onnx-gpt2/gpt2_int8.onnx


## Comparing size

In [16]:
from pathlib import Path

def file_size_mb(path: Path) -> float:
    return path.stat().st_size / (1024 ** 2)

onnx_fp32_size_mb = file_size_mb(onnx_fp32_path)
onnx_int8_size_mb = file_size_mb(onnx_int8_path)

print(f"FP32 ONNX size: {onnx_fp32_size_mb:.2f} MB")
print(f"INT8 ONNX size: {onnx_int8_size_mb:.2f} MB")
print(f"Compression ratio (FP32/INT8): {onnx_fp32_size_mb / onnx_int8_size_mb:.2f}x")
print(f"Size reduction: {(1 - onnx_int8_size_mb / onnx_fp32_size_mb) * 100:.2f}%")

FP32 ONNX size: 475.13 MB
INT8 ONNX size: 119.69 MB
Compression ratio (FP32/INT8): 3.97x
Size reduction: 74.81%


## Saving quantization result

In [17]:
results_dir = Path("../results")
results_dir.mkdir(parents=True, exist_ok=True)

quant_summary_path = results_dir / "onnx_quantization_summary.csv"

summary_df = pd.DataFrame([
    {
        "model": "gpt2-finetuned-onnx-fp32",
        "file_path": str(onnx_fp32_path),
        "size_mb": onnx_fp32_size_mb
    },
    {
        "model": "gpt2-finetuned-onnx-int8",
        "file_path": str(onnx_int8_path),
        "size_mb": onnx_int8_size_mb
    }
])

summary_df.to_csv(quant_summary_path, index=False)


metrics_path = Path("../results/metrics.csv")

if metrics_path.exists():
    metrics_df = pd.read_csv(metrics_path)
else:
    metrics_df = pd.DataFrame()

new_rows = pd.DataFrame([
    {
        "model": "gpt2-finetuned-onnx-fp32",
        "avg_generation_time_sec": np.nan,
        "max_new_tokens": 80,
        "num_return_sequences": 1,
        "approx_model_size_mb": onnx_fp32_size_mb,
    },
    {
        "model": "gpt2-finetuned-onnx-int8",
        "avg_generation_time_sec": np.nan,
        "max_new_tokens": 80,
        "num_return_sequences": 1,
        "approx_model_size_mb": onnx_int8_size_mb,
    },
])

metrics_df = pd.concat([metrics_df, new_rows], ignore_index=True)
metrics_df.to_csv(metrics_path, index=False)

print("Updated:", metrics_path)
metrics_df.tail(6)

Updated: ../results/metrics.csv


,model,avg_generation_time_sec,max_new_tokens,num_return_sequences,approx_model_size_mb
0,gpt2-baseline,1.295833,80,1,474.700195
1,gpt2-finetuned,1.350325,80,1,474.700195
2,gpt2-finetuned-onnx-fp32,NaN,80,1,475.125290
3,gpt2-finetuned-onnx-int8,NaN,80,1,119.687916
